In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np


model = YOLO("yolo26n-pose.pt")

input_video = "input.mp4"
output_black = "pose_output.mp4"
output_overlay = "pose_overlay.mp4"

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise ValueError("Error: Could not open input video!")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0 or width <= 0 or height <= 0:
    cap.release()
    raise ValueError("Error: Invalid video properties!")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out_black = cv2.VideoWriter(
    output_black, fourcc, fps, (width, height)
)

out_overlay = cv2.VideoWriter(
    output_overlay, fourcc, fps, (width, height)
)

if not out_black.isOpened() or not out_overlay.isOpened():
    cap.release()
    out_black.release()
    out_overlay.release()
    raise ValueError("Error: Could not create output videos!")

skeleton = [
    (5, 7), (7, 9),       
    (6, 8), (8, 10),     
    (5, 6),               
    (5, 11), (6, 12),    
    (11, 12),            
    (11, 13), (13, 15),  
    (12, 14), (14, 16),   
    (0, 1), (0, 2),       
    (1, 3), (2, 4),       
    (0, 5), (0, 6)        
]

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, conf=0.25, verbose=False)

    black_frame = np.zeros(
        (height, width, 3), dtype=np.uint8
    )

    overlay_frame = frame.copy()

    keypoints = results[0].keypoints

    if keypoints is not None and keypoints.xy is not None:
        points = keypoints.xy.cpu().numpy()

        if keypoints.conf is not None:
            scores = keypoints.conf.cpu().numpy()
        else:
            scores = np.ones(points.shape[:2])

        for person_points, person_scores in zip(points, scores):

            for start, end in skeleton:

                x1, y1 = person_points[start]
                x2, y2 = person_points[end]

                if (
                    person_scores[start] > 0.25
                    and person_scores[end] > 0.25
                ):
                    pt1 = (int(x1), int(y1))
                    pt2 = (int(x2), int(y2))

                    cv2.line(
                        black_frame,
                        pt1,
                        pt2,
                        (255, 255, 255),
                        3
                    )

                    cv2.line(
                        overlay_frame,
                        pt1,
                        pt2,
                        (255, 255, 255),
                        3
                    )

            for (x, y), score in zip(
                person_points, person_scores
            ):
                if score > 0.25:
                    point = (int(x), int(y))

                    cv2.circle(
                        black_frame,
                        point,
                        4,
                        (255, 255, 255),
                        -1
                    )

                    cv2.circle(
                        overlay_frame,
                        point,
                        4,
                        (255, 255, 255),
                        -1
                    )

    out_black.write(black_frame)
    out_overlay.write(overlay_frame)

cap.release()
out_black.release()
out_overlay.release()

print("Processing completed successfully!")
print("1. Black background skeleton: pose_output.mp4")
print("2. Original video with skeleton: pose_overlay.mp4")


Processing completed successfully!
1. Black background skeleton: pose_output.mp4
2. Original video with skeleton: pose_overlay.mp4
